In [1]:
import pandas as pd

In [2]:
file_name = "Ground Truth Collection Solidaridad Uganda 2026 February"
local_path = "~/Sites/acorn-dqm-streamlit/version2/data/"
partner = "SOLU"
country = "Uganda"

max_plot_area_size: int = 30
min_subplot_area_size: float = 450
max_subplot_area_size: float = 750
max_vertices: int = 4
crs = "EPSG:4326"

In [3]:
path = f"{file_name}.xlsx"

plots_df = pd.read_excel(
    f"{local_path}{file_name}.xlsx",
    sheet_name=0,
)
subplots_df = pd.read_excel(
    f"{local_path}{file_name}.xlsx",
    sheet_name=1,
).rename(
    columns={
        "KEY": "SUBPLOT_KEY",
    }
)

In [4]:
subplots_df

,gt_subplot,subplot_comments,SET-OF-new_vegetation,PARENT_KEY,SUBPLOT_KEY,SET-OF-sub_plot
0,1.0390425 34.3628138 1468.0 3.1;1.0390425 34.3...,"Only coffee and bananas, no other tree species",uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...
1,1.0392676 34.362872 1463.0 4.4;1.0392676 34.36...,Only coffee and bananas on the plot,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...
2,1.0394101 34.3629696 1446.0 1.5;1.0394101 34.3...,NaN,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...
3,1.0396126 34.3630109 1440.0 2.6;1.0396132 34.3...,NaN,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...
4,1.0397926 34.3632681 1430.0 4.5;1.0397925 34.3...,NaN,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...,uuid:32abda65-3bdd-46e9-85d5-5aebe9daef82/sub_...
...,...,...,...,...,...,...
3675,1.068831 34.2540128 1986.0 8.416;1.0690347 34....,NaN,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...
3676,1.0689609 34.2536988 1955.0 6.75;1.0690376 34....,NaN,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...
3677,1.0691545 34.2536028 1968.0 5.0;1.0694043 34.2...,NaN,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...
3678,1.069258 34.253827 1970.0 5.5;1.0692719 34.254...,NaN,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...,uuid:17ee0359-88fe-4a6f-8377-41b419f2c979/sub_...


In [5]:
import pandas as pd
import geopandas as gpd
import shapely
from typing import Callable, Optional, Union
import json
import dataclasses as dc
from shapely.lib import ShapelyError
from pyproj import CRS, Geod, Transformer

from shapely.geometry import (
    GeometryCollection,
    MultiPolygon,
    Point,
    Polygon,
    shape,
)
import math

import utm
from geojson_rewind import rewind
from shapely.geometry.polygon import orient

In [6]:
class GeometryTypeError(ShapelyError):
    """
    An error raised when the type of the geometry in question is
    unrecognized or inappropriate.
    """


def mapping(ob):
    """
    Returns a GeoJSON-like mapping from a Geometry or any
    object which implements __geo_interface__

    Parameters
    ----------
    ob :
        An object which implements __geo_interface__.

    Returns
    -------
    dict

    Examples
    --------
    >>> pt = Point(0, 0)
    >>> mapping(pt)
    {'type': 'Point', 'coordinates': (0.0, 0.0)}
    """
    return ob.__geo_interface__


def transform(func, geom):
    """Applies `func` to all coordinates of `geom` and returns a new
    geometry of the same type from the transformed coordinates.

    `func` maps x, y, and optionally z to output xp, yp, zp. The input
    parameters may iterable types like lists or arrays or single values.
    The output shall be of the same type. Scalars in, scalars out.
    Lists in, lists out.

    For example, here is an identity function applicable to both types
    of input.

      def id_func(x, y, z=None):
          return tuple(filter(None, [x, y, z]))

      g2 = transform(id_func, g1)

    Using pyproj >= 2.1, this example will accurately project Shapely geometries:

      import pyproj

      wgs84 = pyproj.CRS('EPSG:4326')
      utm = pyproj.CRS('EPSG:32618')

      project = pyproj.Transformer.from_crs(wgs84, utm, always_xy=True).transform

      g2 = transform(project, g1)

    Note that the always_xy kwarg is required here as Shapely geometries only support
    X,Y coordinate ordering.

    Lambda expressions such as the one in

      g2 = transform(lambda x, y, z=None: (x+1.0, y+1.0), g1)

    also satisfy the requirements for `func`.
    """
    if geom.is_empty:
        return geom
    if geom.geom_type in ("Point", "LineString", "LinearRing", "Polygon"):
        # First we try to apply func to x, y, z sequences. When func is
        # optimized for sequences, this is the fastest, though zipping
        # the results up to go back into the geometry constructors adds
        # extra cost.
        try:
            if geom.geom_type in ("Point", "LineString", "LinearRing"):
                return type(geom)(zip(*func(*zip(*geom.coords))))
            elif geom.geom_type == "Polygon":
                shell = type(geom.exterior)(zip(*func(*zip(*geom.exterior.coords))))
                holes = list(type(ring)(zip(*func(*zip(*ring.coords)))) for ring in geom.interiors)
                return type(geom)(shell, holes)

        # A func that assumes x, y, z are single values will likely raise a
        # TypeError, in which case we'll try again.
        except TypeError:
            if geom.geom_type in ("Point", "LineString", "LinearRing"):
                return type(geom)([func(*c) for c in geom.coords])
            elif geom.geom_type == "Polygon":
                shell = type(geom.exterior)([func(*c) for c in geom.exterior.coords])
                holes = list(type(ring)([func(*c) for c in ring.coords]) for ring in geom.interiors)
                return type(geom)(shell, holes)

    elif geom.geom_type.startswith("Multi") or geom.geom_type == "GeometryCollection":
        return type(geom)([transform(func, part) for part in geom.geoms])
    else:
        raise GeometryTypeError(f"Type {geom.geom_type!r} not recognized")


def round_coordinates(geom, ndigits=2):
    def _round_coords(x, y, z=None):
        x = round(x, ndigits)
        y = round(y, ndigits)
        if z is not None:
            z = round(z, ndigits)
        return [c for c in (x, y, z) if c is not None]

    return transform(_round_coords, geom)


def to_geojson(geom: Polygon, id: str = "plot_id") -> Optional[str]:
    if geom is None:
        return None
    else:
        return json.dumps(
            {
                "type": "FeatureCollection",
                "features": [
                    {
                        "type": "Feature",
                        "properties": {"id": id},
                        "geometry": mapping(round_coordinates(geom, 7)),
                    }
                ],
            }
        )


def length_width_ratio(geom: Polygon, geodisic=False) -> Optional[float]:
    if geom.is_empty:
        return None

    if not geodisic:
        geom = geom_to_utm(geom)
    mbb = geom.minimum_rotated_rectangle
    x, y = mbb.exterior.coords.xy

    if geodisic:
        geod = Geod(ellps="WGS84")
        edge_length = (
            geod.inv(x[0], y[0], x[1], y[1])[2],
            geod.inv(x[1], y[1], x[2], y[2])[2],
        )
    else:
        edge_length = (
            Point(x[0], y[0]).distance(Point(x[1], y[1])),
            Point(x[1], y[1]).distance(Point(x[2], y[2])),
        )

    if min(edge_length) == 0:
        print("too small width, length is: " + str(max(edge_length)))
        ratio = max(edge_length) / 0.00000000001  # very small number
    else:
        ratio = max(edge_length) / min(edge_length)
    return ratio


def add_length_width_ratio(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf["length_width_ratio"] = gdf.geometry.apply(length_width_ratio, geodisic=True)
    return gdf


def calculate_minimum_rotated_rectangle(gdf: gpd.GeoDataFrame, geodesic=False) -> gpd.GeoDataFrame:
    if not geodesic:
        gdf["minimum_rotated_rectangle_m2"] = gdf.geometry.apply(
            lambda x: geom_to_utm(x).minimum_rotated_rectangle.area
        )
    else:
        geod = Geod(ellps="WGS84")
        gdf["minimum_rotated_rectangle_m2"] = gdf.geometry.apply(
            lambda x: abs(geod.geometry_area_perimeter(x.minimum_rotated_rectangle)[0])
        )
    return gdf


def add_protruding_ratio(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf_shapes = (
        gdf[~gdf.geometry.isna()]
        .pipe(calculate_area, geodisic=True)
        .pipe(calculate_minimum_rotated_rectangle, geodesic=True)
        .assign(mrr_ratio=lambda x: x.minimum_rotated_rectangle_m2 / x.area_m2)
        .drop(columns=["area_m2", "minimum_rotated_rectangle_m2"])
    )
    return gdf_shapes


def calculate_area(
    gdf: gpd.GeoDataFrame,
    geodisic=False,
    area_field: str = "area_m2",
    geometry_field: str = "geometry",
) -> gpd.GeoDataFrame:
    """Calculates area and adds it to a new column in the gdf.

    Args:
        geodesic: If True, it will calculate the area in geodesic coordinates
        area_field: Name of field it will add
        geometry_field: Name of field on which it will calculate area

    Returns:
        gdf: original geodataframe with area column added (float dtype)
    """

    # this is needed, otherwise it becomes a geometry column
    if len(gdf) == 0:
        gdf[area_field] = None
        gdf[area_field] = gdf[area_field].astype(float)
        return gdf

    if geodisic:
        if gdf.crs.to_epsg() != 4326:
            gdf_4326 = gdf.to_crs(crs)
        else:
            gdf_4326 = gdf

        geod = Geod(ellps="WGS84")

        gdf[area_field] = gdf_4326[geometry_field].apply(
            lambda x: (0 if x is None or pd.isna(x) else abs(geod.geometry_area_perimeter(x)[0]))
        )

    else:
        if gdf.crs.is_projected:
            gdf[area_field] = gdf[geometry_field].area
        else:
            gdf[area_field] = gdf[geometry_field].apply(
                lambda x: (0 if x is None or pd.isna(x) else geom_to_utm(x).area)
            )

    return gdf


def fix_self_intersecting_square(geom: Polygon) -> Polygon:
    if geom is None:
        return geom
    if geom.is_valid:
        return geom
    coords = geom.exterior.coords.xy
    if len(coords[0]) == 5:
        return geom.convex_hull
    return geom


def fix_with_orient(geom: Polygon) -> Polygon:
    if geom is None:
        return geom
    if geom.is_valid:
        return geom
    new_geom = orient(geom)
    if new_geom.is_valid:
        return new_geom
    return geom


def fix_with_rewind(geom: Polygon) -> Polygon:
    if geom is None:
        return geom
    if geom.is_valid:
        return geom
    new_geom = shape(json.loads(rewind(json.dumps(mapping(geom)))))
    if new_geom.is_valid:
        return new_geom
    return geom


def fix_with_zero_buffer(geom: Polygon) -> Polygon:
    if geom is None:
        return None
    elif geom.is_valid:
        return geom
    new_geom = geom.buffer(0)
    if geom.area > 0:
        ratio = new_geom.area / geom.area
    else:
        ratio = 10000
    if 0.995 <= ratio and ratio < 1.005:
        return new_geom
    return geom


def fix_with_2d_polygon(geom: Polygon) -> Polygon:
    if geom is None:
        return Polygon()
    else:
        return shapely.wkb.loads(shapely.wkb.dumps(geom, output_dimension=2))


def geom_to_utm(geom: Polygon) -> Polygon:
    if geom.is_empty:
        return geom
    lon, lat = geom.centroid.x, geom.centroid.y
    if not -80.0 <= lat <= 84.0:
        # change/mirror XY-coordinates?
        geom = Polygon()
        return geom
    if not -180.0 <= lon <= 180.0:
        # change/mirror XY-coordinates?
        geom = Polygon()
        return geom
    _, _, zone, _ = utm.from_latlon(lat, lon)
    project = Transformer.from_crs(
        CRS(crs),
        CRS.from_dict({"proj": "utm", "zone": zone, "south": lat < 0}),
        always_xy=True,
    ).transform
    return transform(project, geom)


def nr_vertices(geom: Polygon) -> Optional[int]:
    if geom is None or geom.is_empty:
        return 0
    elif isinstance(geom, Polygon):
        return len(geom.exterior.coords.xy[0])
    else:
        return None


def number_of_vertices_per_polygon(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf["nr_vertices"] = gdf.geometry.apply(nr_vertices)
    return gdf


def remove_duplicate_vertices(geom: Polygon) -> Polygon:
    if geom.is_empty:
        return Polygon()
    else:
        coords_list = list(geom.exterior.coords)
        if ~len(coords_list) == 0:
            if coords_list[0] == coords_list[-1]:
                coords_list = coords_list[:-1]
            unique_vertices = list(set(coords_list))
            if len(unique_vertices) < len(coords_list):
                unique_vertices.append(unique_vertices[0])
                if len(unique_vertices) > 2:
                    return Polygon((unique_vertices))
                else:
                    return Polygon()
        return geom


def replace_area_zero(geom: Polygon) -> Polygon:
    if geom is None:
        return None
    elif geom.area == 0 and geom.is_empty:
        return Polygon()
    else:
        return geom


def replace_invalid(geom: Polygon) -> Polygon:
    if geom.is_valid:
        return geom
    elif geom.buffer(0).is_empty:
        return geom.representative_point().buffer(0.00005)
    else:
        return Polygon()


def replace_multipolygons(
    geom: Union[Polygon, MultiPolygon, GeometryCollection],
) -> Union[Polygon, MultiPolygon, GeometryCollection]:
    if isinstance(geom, Polygon):
        return geom
    elif isinstance(geom, MultiPolygon):
        # perform fixes
        if len(geom.geoms) == 1:
            return geom.geoms[0]  # type: ignore
        else:
            total_area_geom = geom_to_utm(geom).area
            max_area_geom = max(geom.geoms, key=lambda x: geom_to_utm(x).area)
            max_area = geom_to_utm(max_area_geom).area
            if 0.9 * total_area_geom < max_area:
                return max_area_geom
            else:
                return geom

    elif isinstance(geom, GeometryCollection):
        if len(geom.geoms) == 1:
            print(geom)
            return geom.geoms[0]  # type: ignore
        else:
            return geom

    else:
        raise SyntaxError("Geometry is not a Polygon, Multipolygon or GeometryCollection")


def replace_none_geometries(geom: Polygon) -> Polygon:
    if geom is None:
        return Polygon()
    return geom


def replace_out_of_bound_geometries(geom: Polygon) -> Polygon:
    if geom.is_empty:
        return geom
    minx, miny, maxx, maxy = geom.bounds
    if minx <= -180 or 180 <= maxx or miny <= -80 or 84 <= maxy:
        return Polygon()
    return geom


def geom_to_utm_with_crs(geom: Polygon) -> Polygon:
    if geom.is_empty:
        return geom
    lon, lat = geom.centroid.x, geom.centroid.y
    if not -80.0 <= lat <= 84.0:
        # change/mirror XY-coordinates?
        geom = Polygon()
        return geom
    if not -180.0 <= lon <= 180.0:
        # change/mirror XY-coordinates?
        geom = Polygon()
        return geom
    _, _, zone, _ = utm.from_latlon(lat, lon)
    crs_dict = CRS.from_dict({"proj": "utm", "zone": zone, "south": lat < 0})
    project = Transformer.from_crs(CRS(crs), crs_dict, always_xy=True).transform
    geom_utm = transform(project, geom)
    return geom_utm, crs_dict


def simplify_geometry(geom, tolerance: float = 0.05, units="meters"):
    """simplifies geometry using tolerance either in meters
    (for which it will transform geom to UTM -SLOW!) or degrees"""
    assert units in ("meters", "degrees")
    if geom.is_empty:
        return geom

    if units == "meters":
        utm_geom_with_crs = geom_to_utm_with_crs(geom)
        if type(utm_geom_with_crs) is tuple:
            utm_geom, utm_crs_dict = utm_geom_with_crs[0], utm_geom_with_crs[1]
        else:
            return utm_geom_with_crs
        utm_geom_s = utm_geom.simplify(tolerance, preserve_topology=True)
        project = Transformer.from_crs(utm_crs_dict, CRS(crs), always_xy=True).transform
        simple_geom = transform(project, utm_geom_s)
    else:
        simple_geom = geom.simplify(tolerance, preserve_topology=True)

    if ~simple_geom.is_valid:
        return geom

    return simple_geom


@dc.dataclass(frozen=True)
class WGS84Point:
    """
    A point on earth in WGS 84 (EPSG 4326).

    Important:
        It is the constructors responsibility to ensure a point is valid. Specifically
        this means that ``-90 <= latitude <= 90 and -180 <= longitude <= 180``.

    Raises:
        ValueError: If ``latitude`` or ``longitude`` is out of bounds.

    See Also:
        See https://epsg.io/4326 for more info on the coordinate system.
    """

    latitude: float
    longitude: float

    def __post_init__(self) -> None:
        if not (-90.0 <= self.latitude <= 90.0):
            raise ValueError(f"latitude should be between -90 and 90 but found {self.latitude!r}")

        if not (-180.0 <= self.longitude <= 180.0):
            raise ValueError(f"longitude should be between -180 and 180 but found {self.longitude!r}")


def gdf_center(gdf: gpd.GeoDataFrame) -> WGS84Point:
    min_lon, min_lat, max_lon, max_lat = gdf.geometry.total_bounds
    lat = (min_lat + max_lat) / 2
    lon = (min_lon + max_lon) / 2
    return WGS84Point(latitude=lat, longitude=lon)


def epsg_code(lon: float, lat: float) -> str:
    utm_band = str((math.floor((lon + 180) / 6) % 60) + 1)
    if len(utm_band) == 1:
        utm_band = "0" + utm_band
    if lat >= 0:
        epsg_code = "326" + utm_band
        return epsg_code
    epsg_code = "327" + utm_band
    return epsg_code


def wgs_to_utm(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf.crs = crs
    centroid = gdf_center(gdf)
    code = epsg_code(centroid.longitude, centroid.latitude)
    return gdf.to_crs(f"EPSG:{code}")


def validate_overlap(
    gdf: gpd.GeoDataFrame,
    id_column: str,
    min_overlap: float,
    buffer: float = -5,
    filter: Callable[[gpd.GeoDataFrame], gpd.GeoDataFrame] = lambda x: x,
) -> gpd.GeoDataFrame:
    df_overlap = (
        gdf.pipe(lambda x: wgs_to_utm(x))
        .assign(geometry=lambda x: x.geometry.buffer(buffer))
        .to_crs(crs)
        .pipe(filter)
        .pipe(calculate_area, geodisic=True)
        .pipe(lambda x: x.overlay(x, keep_geom_type=False))
        .pipe(lambda x: x[x[f"{id_column}_1"] != x[f"{id_column}_2"]])
        .pipe(calculate_area, geodisic=True)
        .assign(
            min_area=lambda x: (
                x[["area_m2_1", "area_m2_2"]].min(axis=1) if not x.empty else pd.Series(dtype="float64")
            ),
            overlay_ratio=lambda x: x.area_m2 / x.min_area,
            overlap=lambda x: min_overlap < x.overlay_ratio.to_numpy(),
        )
        .pipe(lambda x: x[x.overlap])
    )
    if df_overlap.empty:
        df_overlap_all = gpd.GeoDataFrame(columns=[id_column, "overlap_ids", "percentage_overlap"])
    else:
        df_overlap_ids = (
            df_overlap.groupby(f"{id_column}_1")[f"{id_column}_2"]
            .apply(lambda x: ";".join([str(i) for i in x if i is not None]))
            .reset_index()
            .rename(columns={f"{id_column}_1": id_column, f"{id_column}_2": "overlap_ids"})
        )

        df_overlap_max = (
            df_overlap.groupby(f"{id_column}_1")
            .agg({"overlay_ratio": ["max"]})
            .droplevel(axis=1, level=0)
            .reset_index()
            .rename(columns={f"{id_column}_1": id_column, "max": "percentage_overlap"})
            .round(decimals=2)
        )

        df_overlap_all = df_overlap_ids.merge(df_overlap_max, on=id_column, how="left")
    return gdf.merge(df_overlap_all, how="left").assign(
        overlap_ids=lambda x: x.overlap_ids.fillna(""),
        percentage_overlap_float=lambda x: x.percentage_overlap.astype(float),
        percentage_overlap=lambda x: x.percentage_overlap_float.fillna(""),
    )


def total_nr_vertices(gdf):
    """
    Calculate the total number of vertices in a GeoDataFrame.
    """
    return gdf.geometry.apply(lambda x: len(x.exterior.coords) if x.geom_type == "Polygon" else 0).sum()


def is_invalid_polygon_string(polygon_string, pd_row=None, column=None):
    if pd.isna(polygon_string):
        if column is not None:
            print(f"No coordinates for :{column}, so returning empty Polygon")
        return True
    if len(polygon_string) == 32767:
        if pd_row is not None:
            print(
                f"Reached cell limit of excel for: {getattr(pd_row, 'plot_id', '')} "
                f"collected by: {getattr(pd_row, 'enumerator_id', '')},"
                "so returning empty Polygon"
            )
        return True
    return False


def coordinates_from_vertices(vertices, accuracy_m, accuracy_zero_valid=False):
    coordinates = []
    skip_coordinates_counter = 0

    for vertex in vertices:
        if not vertex.strip():
            skip_coordinates_counter += 1
            continue

        parts = vertex.strip().split(" ")
        accuracy = float(parts[3])

        if accuracy_zero_valid:
            if accuracy > accuracy_m:
                skip_coordinates_counter += 1
                continue
        else:
            if accuracy > accuracy_m or abs(accuracy - 0.0) < 1e-9:
                skip_coordinates_counter += 1
                continue

        lon = float(parts[0])
        lat = float(parts[1])
        coordinates.append((lat, lon))
    return coordinates, skip_coordinates_counter


def geom_from_scto_str(pd_row, column, accuracy_m, accuracy_zero_valid=False):
    polygon_string = pd_row[column]
    if is_invalid_polygon_string(polygon_string, pd_row, column):
        return shapely.geometry.polygon.Polygon()

    vertices = polygon_string.split(";")
    coordinates, skip_coordinates_counter = coordinates_from_vertices(vertices, accuracy_m, accuracy_zero_valid)

    if len(coordinates) < 3 or (len(coordinates) < (skip_coordinates_counter * 4)):
        print("Dropped too many points for pd_row")
        return shapely.geometry.polygon.Polygon()
    geom = shapely.geometry.polygon.Polygon(coordinates)
    return geom


def collect_reasons(row: pd.Series) -> str:
    if row.geometry is None:
        return "Geometry missing"
    if row.geometry.is_empty:
        return "Empty geometry"
    if not row.geometry.is_valid:
        return "Invalid geometry"

    reasons = [
        "Overlapping polygons" if "overlap_ids" in row and row.overlap_ids else "",
        "Duplicate plot id" if row.duplicate_id else "",
        "Boundary not in country" if not row.in_country else "",
        "Plot outside of radius" if not row.in_radius else "",
        "Plot too small" if row.area_m2 < min_subplot_area_size else "",
        "Plot too big" if row.area_m2 > max_subplot_area_size else "",
        f"Nr vertices <= {3}" if row.nr_vertices_too_small else "",
        "Plot is protruding" if row.protruding_ratio_too_big else "",
    ]

    return ";".join(filter(None, reasons))


def assign_geom_valid_geojson(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf["reasons"] = gdf.apply(collect_reasons, axis=1)
    gdf["geom_valid"] = gdf["reasons"].apply(lambda x: len(x) == 0)
    gdf["geojson"] = gdf.apply(lambda x: to_geojson(x.geometry, x.subplot_id), axis=1)
    return gdf

In [7]:
class GeometryFixer:
    """Handles geometry fixing operations"""

    def fix_geometry(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        gdf["original_vertices"] = gdf.geometry.apply(
            lambda x: len(x.exterior.coords) if x.geom_type == "Polygon" else 0
        )
        gdf["geometry"] = gdf.geometry.apply(remove_duplicate_vertices)

        valid = gdf.is_valid.sum()

        gdf["geometry"] = gdf.geometry.apply(fix_self_intersecting_square)
        gdf["geometry"] = gdf.geometry.apply(fix_with_orient)
        gdf["geometry"] = gdf.geometry.apply(fix_with_rewind)
        gdf["geometry"] = gdf.geometry.apply(fix_with_zero_buffer)
        gdf["geometry"] = gdf.geometry.apply(fix_with_2d_polygon)
        gdf["geometry"] = gdf["geometry"].apply(
            lambda geom: (geom if geom.geom_type not in ["Point", "LineString", "MultiLineString"] else Polygon())
        )
        gdf["geometry"] = gdf.geometry.apply(replace_multipolygons)
        gdf["geometry"] = gdf.geometry.apply(simplify_geometry, tolerance=0.1)

        print(f"\nFixed {gdf.is_valid.sum() - valid} polygons")
        empty = gdf.is_empty.sum()

        gdf["geometry"] = gdf.geometry.apply(replace_area_zero)
        gdf["geometry"] = gdf.geometry.apply(replace_none_geometries)
        gdf["geometry"] = gdf.geometry.apply(replace_out_of_bound_geometries)
        gdf["geometry"] = gdf.geometry.apply(replace_invalid)
        print(f"Replaced {gdf.is_empty.sum() - empty} polygons with empty polygons\n")
        return gdf


class GeometryValidator:
    """Handles geometry validation operations"""

    def __init__(
        self,
        partner: str,
        country: str,
        threshold_length_width: float,
        threshold_protruding_ratio: float,
        validate_id: str,
        threshold_within_radius: float,
        min_area_size: float,
        max_area_size: float,
        max_vertices: float,
    ):
        self.partner = partner
        self.country = country
        self.threshold_length_width = threshold_length_width
        self.threshold_protruding_ratio = threshold_protruding_ratio
        self.validate_id = validate_id
        self.threshold_within_radius = threshold_within_radius
        self.min_area_size = min_area_size
        self.max_area_size = max_area_size
        self.max_vertices = max_vertices

    def validate_geometry(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        if "index" in gdf.columns:
            gdf = gdf.drop(columns=["index"])

        if gdf.geometry.is_empty.all():
            print("All geometries in gdf are empty, no validation can be done")
        else:
            gdf = (
                gdf.pipe(self.validate_length_width_ratio)
                .pipe(self.validate_protruding_ratio)
                .assign(in_country=True)
                .assign(duplicate_id=False)
                .pipe(calculate_area, geodisic=True)
                .pipe(self.validate_nr_vertices)
                .pipe(self.validate_within_radius)
                .pipe(
                    validate_overlap,
                    id_column=self.validate_id,
                    min_overlap=0.5,
                    buffer=-5,
                    filter=self.overlap_filter,
                )
            )

        return gdf

    def validate_length_width_ratio(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        return gdf.pipe(add_length_width_ratio).assign(
            length_width_ratio_too_big=lambda x: x.length_width_ratio > self.threshold_length_width
        )

    def validate_protruding_ratio(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        return gdf.pipe(add_protruding_ratio).assign(
            protruding_ratio_too_big=lambda x: x.mrr_ratio > self.threshold_protruding_ratio
        )

    def validate_nr_vertices(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        return (
            gdf.pipe(number_of_vertices_per_polygon)
            .assign(vertices_dropped=lambda x: x.original_vertices - x.nr_vertices)
            .assign(vertices_valid_percentage=lambda x: 100 * x.nr_vertices / x.original_vertices)
            .assign(nr_vertices_too_small=lambda x: x.nr_vertices <= self.max_vertices)
        )

    def validate_within_radius(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        gdf["in_radius"] = gdf.geometry.apply(lambda x: self.all_points_in_radius(x, self.threshold_within_radius))
        return gdf

    def overlap_filter(self, x):
        return x[
            (x.in_country)
            & (x.in_radius)
            & (self.min_area_size < x.area_m2)
            & (x.area_m2 < self.max_area_size)
            & (~x.protruding_ratio_too_big)
            & (~x.nr_vertices_too_small)
        ]

    @staticmethod
    def all_points_in_radius(geom: Polygon, radius: float):
        if geom is not None and geom.is_valid:
            geom_utm = geom_to_utm(geom)
            circle = geom_utm.centroid.buffer(radius)
            return circle.covers(geom_utm)
        return None

In [8]:
geometry_validator = GeometryValidator(
    partner=partner,
    country=country,
    threshold_length_width=2,
    threshold_protruding_ratio=1.55,
    validate_id="subplot_id",
    threshold_within_radius=40,
    min_area_size=min_subplot_area_size,
    max_area_size=max_subplot_area_size,
    max_vertices=max_vertices,
).validate_geometry

geometry_fixer = GeometryFixer().fix_geometry

In [9]:
# raw plots file
plots_df["geometry"] = plots_df.apply(
    lambda row: geom_from_scto_str(row, column="gt_plot", accuracy_m=10, accuracy_zero_valid=False),
    axis=1,
)

gdf_plots = gpd.GeoDataFrame(plots_df, geometry="geometry", crs=4326)

gdf_plots.to_file(
    f"{local_path}{file_name}_plots.geojson",
    driver="GeoJSON",
)

Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row


In [10]:
subplots_df_plot = (
    pd.merge(
        plots_df,
        subplots_df,
        how="outer",
        left_on="KEY",
        right_on="PARENT_KEY",
        validate="1:m",
    ).drop(["KEY", "PARENT_KEY"], axis=1)
)[
    [
        "starttime",
        "enumerator",
        "dbh_method",
        "measured_subplots",
        "gt_subplot",
        "SUBPLOT_KEY",
    ]
].rename(columns={"SUBPLOT_KEY": "subplot_id"})

In [11]:
subplots_df_plot_raw = subplots_df_plot.copy()

subplots_df_plot_raw["geometry"] = subplots_df_plot_raw.apply(
    lambda row: geom_from_scto_str(row, column="gt_subplot", accuracy_m=1000, accuracy_zero_valid=True),
    axis=1,
)

gdf_subplots_raw = gpd.GeoDataFrame(
    subplots_df_plot_raw,
    geometry="geometry",
    crs=4326,
)

gdf_subplots_raw.to_file(
    f"{local_path}{file_name}_subplots_original.geojson",
    driver="GeoJSON",
)

In [12]:
# Subplot geojson with checks and accuracy checks
subplots_df_plot["geometry"] = subplots_df_plot.apply(
    lambda row: geom_from_scto_str(row, column="gt_subplot", accuracy_m=10, accuracy_zero_valid=False),
    axis=1,
)

gdf_subplots = gpd.GeoDataFrame(
    subplots_df_plot,
    geometry="geometry",
    crs=4326,
)

gdf_subplots_fix = gdf_subplots.pipe(geometry_fixer)
gdf_subplots_valid = gdf_subplots_fix.pipe(geometry_validator)
gdf_subplots_validity = gdf_subplots_valid.pipe(assign_geom_valid_geojson)

gdf_subplots_validity.to_file(
    f"{local_path}{file_name}_subplots_checks.geojson",
    driver="GeoJSON",
)

Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many points for pd_row
Dropped too many poi

/var/folders/00/btls1cns2d544mn2rkm3v4v40000gn/T/ipykernel_735/1916655726.py:440: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  if ~simple_geom.is_valid:
/var/folders/00/btls1cns2d544mn2rkm3v4v40000gn/T/ipykernel_735/1916655726.py:440: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  if ~simple_geom.is_valid:
/var/folders/00/btls1cns2d544mn2rkm3v4v40000gn/T/ipykernel_735/1916655726.py:440: DeprecationWarning


Fixed 1561 polygons
Replaced 17 polygons with empty polygons



/Users/galihpratama/Library/Python/3.13/lib/python/site-packages/shapely/constructive.py:1353: RuntimeWarning: invalid value encountered in oriented_envelope
  return lib.oriented_envelope(geometry, **kwargs)
/Users/galihpratama/Library/Python/3.13/lib/python/site-packages/shapely/constructive.py:1353: RuntimeWarning: invalid value encountered in oriented_envelope
  return lib.oriented_envelope(geometry, **kwargs)


In [13]:
gdf_subplots_validity

,starttime,enumerator,dbh_method,measured_subplots,gt_subplot,subplot_id,geometry,original_vertices,length_width_ratio,length_width_ratio_too_big,...,vertices_dropped,vertices_valid_percentage,nr_vertices_too_small,in_radius,overlap_ids,percentage_overlap,percentage_overlap_float,reasons,geom_valid,geojson
0,2026-02-25 19:19:17.702,Assah saviour (131145),circumference,16,1.0382556 34.4207565 1768.0 7.416;1.0382555 34...,uuid:04ab4dc7-ffd1-4f0d-bbe4-b2e2d0c3f2e3/sub_...,"POLYGON ((34.42076 1.03826, 34.42076 1.03826, ...",10,1.517975,False,...,0,100.000000,False,True,,,NaN,,True,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
1,2026-02-25 19:19:17.702,Assah saviour (131145),circumference,16,1.0380601 34.4208931 1758.0 5.1;1.03806 34.420...,uuid:04ab4dc7-ffd1-4f0d-bbe4-b2e2d0c3f2e3/sub_...,"POLYGON ((34.42089 1.03806, 34.42104 1.03826, ...",9,1.337441,False,...,3,66.666667,False,True,,,NaN,,True,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
2,2026-02-25 19:19:17.702,Assah saviour (131145),circumference,16,1.0380064 34.4211474 1759.0 2.9;1.0380064 34.4...,uuid:04ab4dc7-ffd1-4f0d-bbe4-b2e2d0c3f2e3/sub_...,"POLYGON ((34.42115 1.03801, 34.42115 1.03801, ...",13,1.228776,False,...,0,100.000000,False,True,,,NaN,,True,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
3,2026-02-25 19:19:17.702,Assah saviour (131145),circumference,16,1.0379516 34.4213574 1744.0 7.7;1.0379475 34.4...,uuid:04ab4dc7-ffd1-4f0d-bbe4-b2e2d0c3f2e3/sub_...,"POLYGON ((34.42136 1.03795, 34.42145 1.03808, ...",49,1.292020,False,...,4,91.836735,False,True,,,NaN,,True,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
4,2026-02-25 19:19:17.702,Assah saviour (131145),circumference,16,1.0379974 34.4216252 1783.0 6.166;1.0379973 34...,uuid:04ab4dc7-ffd1-4f0d-bbe4-b2e2d0c3f2e3/sub_...,"POLYGON ((34.42153 1.03825, 34.42172 1.03812, ...",31,1.220757,False,...,4,87.096774,False,True,,,NaN,,True,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3675,2026-03-21 16:28:03.287,Chelangat Dolphine (214949),circumference,16,1.2113014 34.3999241 1870.0 4.2;1.211339174148...,uuid:ff9aac58-4f3b-41b4-a25f-c49077ffa8fc/sub_...,POLYGON EMPTY,0,NaN,False,...,0,NaN,True,False,,,NaN,Empty geometry,False,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
3676,2026-03-21 16:28:03.287,Chelangat Dolphine (214949),circumference,16,1.2111494 34.3997923 1871.0 6.666;1.2113886 34...,uuid:ff9aac58-4f3b-41b4-a25f-c49077ffa8fc/sub_...,POLYGON EMPTY,0,NaN,False,...,0,NaN,True,False,,,NaN,Empty geometry,False,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
3677,2026-03-21 16:28:03.287,Chelangat Dolphine (214949),circumference,16,1.2110597 34.4000211 1878.0 2.4;1.211242301000...,uuid:ff9aac58-4f3b-41b4-a25f-c49077ffa8fc/sub_...,POLYGON EMPTY,0,NaN,False,...,0,NaN,True,False,,,NaN,Empty geometry,False,"{""type"": ""FeatureCollection"", ""features"": [{""t..."
3678,2026-03-21 16:28:03.287,Chelangat Dolphine (214949),circumference,16,1.2109937 34.4002464 1876.0 3.833;1.2109821848...,uuid:ff9aac58-4f3b-41b4-a25f-c49077ffa8fc/sub_...,POLYGON EMPTY,0,NaN,False,...,0,NaN,True,False,,,NaN,Empty geometry,False,"{""type"": ""FeatureCollection"", ""features"": [{""t..."


Bad pipe message: %s [b'\x17\x7fgirRb/\x87\x19\x8a\x0f\x88\x85\x1fn\x9bI e%)\x8d\xc7\xefn`5\xb1\xa4\xa3\xf1\xeaB\xa9h', b":{qs\xf1\xb8Q\xa4\xa1Drv3\x00$\x13\x01\x13\x02\x13\x03\xc0/\xc0+\xc00\xc0,\xc0'\xcc\xa9\xcc\xa8\xc0\t\xc0\x13\xc0\n\xc0\x14\x00\x9c\x00\x9d\x00/\x005\x01\x00\x00p\x00\x17\x00\x00\xff\x01\x00\x01\x00\x00\n\x00\x08\x00\x06\x00\x1d\x00\x17\x00\x18\x00\x0b\x00\x02\x01\x00\x00#\x00\x00\x00\r\x00\x14\x00\x12\x04", b'\x04\x04']
Bad pipe message: %s [b'', b'\x05\x05']
Bad pipe message: %s [b'']
Bad pipe message: %s [b'\x01\x02\x01\x003']
